# 🚀 CardioAI: Backend Pipeline (Up to CNN)
This notebook demonstrates the foundational backend pipeline for the Explainable Heart Disease Prediction System. 
**Scope:** Dataset Loading ➡️ Preprocessing ➡️ GASF Transformation ➡️ CNN Training.
*(Note: LSTM, Attention, and XAI layers are omitted in this baseline version).*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from pyts.image import GramianAngularField

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

## 1. Dataset Collection & EDA

In [ ]:
df = pd.read_csv('heart.csv')
display(df.head())
print(df.info())

In [ ]:
# Visualize Target Distribution
sns.countplot(x='target', data=df)
plt.title("Heart Disease Presence (0 = No, 1 = Yes)")
plt.show()

## 2. Preprocessing & Standardization

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Features (First row):")
print(X_scaled[0])

## 3. GASF Transformation (Tabular to Image)

In [ ]:
# Using Gramian Angular Summation Field (GASF) to convert 13 features to 13x13 image
gasf = GramianAngularField(image_size=13, method='summation')
X_gasf = gasf.fit_transform(X_scaled)

# Plot a sample GASF image
plt.figure(figsize=(5, 5))
plt.imshow(X_gasf[0], cmap='rainbow', origin='lower')
plt.title("GASF Image for Patient 1")
plt.colorbar()
plt.show()

## 4. CNN Architecture & Training

In [ ]:
# Reshape for CNN (Samples, Height, Width, Channels)
X_gasf_cnn = np.expand_dims(X_gasf, axis=-1)

X_train, X_test, y_train, y_test = train_test_split(X_gasf_cnn, y, test_size=0.2, random_state=42)
print(f"Training data shape: {X_train.shape}")

In [ ]:
model = Sequential([
    Conv2D(16, (3, 3), activation='relu', input_shape=(13, 13, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=30, batch_size=16, validation_data=(X_test, y_test))

## 5. Evaluation

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Accuracy')
plt.legend()
plt.show()

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, cmap='Blues', fmt='g')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()